In [46]:
import os
# Suppress unnecessary debug logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU operational:", gpus)
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected")

No GPU detected


In [47]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for:", gpus)
    except RuntimeError as e:
        print("Error configuring memory growth:", e)

In [48]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [49]:
df = pd.read_csv('qoute_dataset.csv')

In [50]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [51]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [52]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [53]:
quotes = quotes.str.lower()
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices, harry, that show what we t...
2    “there are only two ways to live your life. on...
3    “the person, be it gentleman or lady, who has ...
4    “imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [54]:
df.shape

(3038, 2)

In [55]:
import string 
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

In [56]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [57]:
wordindex = tokenizer.word_index
print(len(wordindex))
list(wordindex.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [58]:
sequences = tokenizer.texts_to_sequences(quotes)
print(quotes[0])
print(sequences[0])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [59]:
x = []
y= []

for seq in sequences:
    for i in range(1, len(seq)):
        x.append(seq[:i])
        y.append(seq[i])

In [69]:
max_len = 100


Padding

In [70]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
xpadded = pad_sequences(x, maxlen=max_len, padding='pre')
y = np.array(y)

In [71]:
xpadded.shape, y.shape

((85271, 100), (85271,))

Onehot encoding 

In [72]:
from tensorflow.keras.utils import to_categorical
y_train = np.array(y)

RNN

In [73]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN, Input

embedding_dim = 50
rnn_units = 128

rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units, return_sequences=False))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))
rnn_model.compile(
    loss="sparse_categorical_crossentropy",  # Note the 'sparse_' prefix
    optimizer="adam",
    metrics=["accuracy"],
)

In [74]:
rnn_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

LSTM

In [75]:
lstm_model = Sequential()
lstm_model.add(Input(shape=(max_len,)))
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
lstm_model.add(LSTM(units=rnn_units, return_sequences=False))
lstm_model.add(Dense(units=vocab_size, activation="softmax"))

lstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

lstm_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 100, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,881,648 (7.18 MB)

 Trainable params: 1,881,648 (7.18 MB)

 Non-trainable params: 0 (0.00 B)

In [67]:
epochs = 10
batch_size = 128

history_rnn = rnn_model.fit(xpadded, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.1)


Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 88s 144ms/step - accuracy: 0.0441 - loss: 6.7163 - val_accuracy: 0.0522 - val_loss: 6.5735
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 85s 142ms/step - accuracy: 0.0757 - loss: 6.1383 - val_accuracy: 0.0889 - val_loss: 6.3808
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 84s 140ms/step - accuracy: 0.1037 - loss: 5.7663 - val_accuracy: 0.1020 - val_loss: 6.3319
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 92s 153ms/step - accuracy: 0.1189 - loss: 5.4665 - val_accuracy: 0.1068 - val_loss: 6.3465
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 91s 151ms/step - accuracy: 0.1315 - loss: 5.2059 - val_accuracy: 0.1103 - val_loss: 6.3913
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 89s 148ms/step - accuracy: 0.1435 - loss: 4.9697 - val_accuracy: 0.1109 - val_loss: 6.4604
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 88s 146ms/step - accuracy: 0.1577 - loss: 4.7504 - val_accuracy: 0.1099 - val_loss: 6.5221
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 89s 148ms/step - accuracy: 0.1713 - loss: 4

In [76]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop if validation loss doesn't improve for 5 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Save the best model weights
checkpoint = ModelCheckpoint(
    'best_quote_lstm.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    xpadded,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.0392 - loss: 6.7754
Epoch 1: val_loss improved from None to 6.68230, saving model to best_quote_lstm.keras

Epoch 1: finished saving model to best_quote_lstm.keras
600/600 ━━━━━━━━━━━━━━━━━━━━ 53s 87ms/step - accuracy: 0.0392 - loss: 6.7754 - val_accuracy: 0.0474 - val_loss: 6.6823
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.0575 - loss: 6.3361
Epoch 2: val_loss improved from 6.68230 to 6.58074, saving model to best_quote_lstm.keras

Epoch 2: finished saving model to best_quote_lstm.keras
600/600 ━━━━━━━━━━━━━━━━━━━━ 52s 86ms/step - accuracy: 0.0575 - loss: 6.3361 - val_accuracy: 0.0632 - val_loss: 6.5807
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.0777 - loss: 6.0921
Epoch 3: val_loss improved from 6.58074 to 6.48987, saving model to best_quote_lstm.keras

Epoch 3: finished saving model to best_quote_lstm.keras
600/600 ━━━━━━━━━━━━━━━━━━━━ 51s 86ms/step - accuracy: 0.077

In [77]:
import pickle

# 1. Save model architecture and trained weights
lstm_model.save("lstm_quote_model.h5")

# 2. Save tokenizer vocabulary and word index
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# 3. Save max_len so inference padding matches training padding exactly
with open("config.pkl", "wb") as f:
    pickle.dump({"max_len": max_len}, f)

print("Saved: lstm_quote_model.h5, tokenizer.pkl, config.pkl")

Saved: lstm_quote_model.h5, tokenizer.pkl, config.pkl


In [78]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences


def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / max(temperature, 1e-4)
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)


def generate_quote(
    model,
    tokenizer,
    max_len,
    seed_text,
    next_words=20,
    temperature=0.8,
):
    output_text = seed_text
    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len, padding="pre")

        predictions = model.predict(token_list, verbose=0)[0]
        predicted_idx = sample_with_temperature(predictions, temperature)
        predicted_word = index_to_word.get(predicted_idx, "")

        if not predicted_word:
            break

        output_text += " " + predicted_word

    return output_text


# Quick test
test_quote = generate_quote(
    lstm_model,
    tokenizer,
    max_len,
    seed_text="Life is",
    next_words=15,
    temperature=0.7,
)
print("Generated Quote:\n", test_quote)

Generated Quote:
 Life is a lover but for the ability in a courage to be little for us it
